# 🧠 Alzheimer's Disease Detection — Machine Learning Benchmark (Colab)

This notebook trains and evaluates **8 traditional Machine Learning classifiers** on the graph features extracted from the Augmented Alzheimer MRI Dataset. 

It compiles a complete comparative analysis of **CNN vs. GAT (Ours) vs. ML Baselines**, generating publication-quality tables and comparative charts.

In [ ]:
# ---- Install required dependencies ----
!pip install -q torch-geometric xgboost lightgbm openpyxl scikit-learn pandas matplotlib seaborn

In [ ]:
# ---- Imports & Config ----
import os
import time
import glob
import random
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from google.colab import files

import torch
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as GeoDataLoader
from torch_geometric.nn import global_mean_pool

from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

IMG_SIZE = 224
PATCH_GRID = 14
PATCH_SIZE = IMG_SIZE // PATCH_GRID  # 16x16 pixels
NODE_DIM = 128
KNN_K = 8
BATCH_SIZE = 32

### 🔑 Kaggle Dataset Download
Upload your `kaggle.json` credentials to automatically download the dataset.

In [ ]:
# Upload kaggle.json key
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Please upload your kaggle.json API key:")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    for fn in uploaded.keys():
        !mv "{fn}" ~/.kaggle/kaggle.json
    !chmod 600 ~/.kaggle/kaggle.json

# Download dataset
if not os.path.exists('/content/augmented-alzheimer-mri-dataset.zip'):
    !kaggle datasets download -d uraninjo/augmented-alzheimer-mri-dataset

In [ ]:
# Unzip dataset
import zipfile
if not os.path.exists('/content/AugmentedAlzheimerDataset'):
    print("Unzipping dataset...")
    with zipfile.ZipFile('/content/augmented-alzheimer-mri-dataset.zip', 'r') as zip_ref:
        zip_ref.extractall('/content/')
    print("Done!")

In [ ]:
# Define paths & discover images
train_dir = '/content/AugmentedAlzheimerDataset/'
test_dir  = '/content/OriginalDataset'

class_names = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']

def discover_images(root_dir):
    paths, labels = [], []
    for label_idx, name in enumerate(class_names):
        folder = os.path.join(root_dir, name)
        exts = ('*.jpg', '*.jpeg', '*.png', '*.bmp')
        for ext in exts:
            for p in glob.glob(os.path.join(folder, ext)):
                paths.append(p)
                labels.append(label_idx)
    return paths, labels

train_all_paths, train_all_labels = discover_images(train_dir)
test_paths, test_labels = discover_images(test_dir)

# Stratified Train / Val split (80/20)
class_paths = {i: [] for i in range(4)}
for p, l in zip(train_all_paths, train_all_labels):
    class_paths[l].append(p)

tr_p, tr_l, va_p, va_l = [], [], [], []
for cls_idx, paths in class_paths.items():
    random.shuffle(paths)
    # Subsample to 50% for benchmark speed consistency
    paths = paths[:len(paths)//2]
    
    n_val = int(len(paths) * 0.2)
    va_p += paths[:n_val]
    va_l += [cls_idx] * n_val
    tr_p += paths[n_val:]
    tr_l += [cls_idx] * len(paths[n_val:])

print(f"Dataset splits: Train: {len(tr_p)} | Val: {len(va_p)} | Test: {len(test_paths)}")

### 🖼️ Graph Construction & Builder

In [ ]:
class MRIGraphBuilder:
    def __init__(self, pca=None):
        self.pca = pca
        
    def _load_image(self, path):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(f"Failed to load: {path}")
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        return img.astype(np.float32) / 255.0
        
    def _extract_patches(self, img):
        patches = []
        for i in range(PATCH_GRID):
            for j in range(PATCH_GRID):
                patch = img[i*PATCH_SIZE:(i+1)*PATCH_SIZE, j*PATCH_SIZE:(j+1)*PATCH_SIZE]
                patches.append(patch.flatten())
        return np.vstack(patches)  # (196, 256)
        
    def build(self, path, label):
        img = self._load_image(path)
        raw_feats = self._extract_patches(img)
        
        # PCA dimension reduction
        x = self.pca.transform(raw_feats) if self.pca else raw_feats
        x_tensor = torch.tensor(x, dtype=torch.float32)
        
        # Edges: Spatial Grid (8-connectivity) + Cosine KNN Edge Construction
        edges = []
        weights = []
        N = PATCH_GRID * PATCH_GRID
        
        # 1. Spatial Grid Edges
        for idx in range(N):
            r, c = idx // PATCH_GRID, idx % PATCH_GRID
            for dr in [-1, 0, 1]:
                for dc in [-1, 0, 1]:
                    if dr == 0 and dc == 0:
                        continue
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < PATCH_GRID and 0 <= nc < PATCH_GRID:
                        nidx = nr * PATCH_GRID + nc
                        edges.append((idx, nidx))
                        weights.append(1.0)  # default spatial weight
                        
        # 2. Cosine Similarity k-NN Edges
        norms = np.linalg.norm(x, axis=1, keepdims=True) + 1e-8
        normalized = x / norms
        similarity = np.dot(normalized, normalized.T)
        
        for idx in range(N):
            sim_scores = similarity[idx]
            sim_scores[idx] = -1.0  # skip self-loop
            nearest_indices = np.argsort(sim_scores)[::-1][:KNN_K]
            for nidx in nearest_indices:
                edges.append((idx, nidx))
                weights.append(float(sim_scores[nidx]))
                
        edge_index = torch.tensor(np.array(edges).T, dtype=torch.long)
        edge_attr = torch.tensor(np.array(weights), dtype=torch.float32).unsqueeze(1)
        
        return Data(x=x_tensor, edge_index=edge_index, edge_attr=edge_attr, y=torch.tensor(label, dtype=torch.long))

In [ ]:
# ---- Fit PCA on sample of training images ----
PCA_SAMPLE = 500
print(f"Fitting PCA on {PCA_SAMPLE} training images...")
sample_paths = random.sample(tr_p, min(PCA_SAMPLE, len(tr_p)))
tmp = MRIGraphBuilder(pca=None)
all_patches = []
for path in sample_paths:
    try:
        img = tmp._load_image(path)
        all_patches.append(tmp._extract_patches(img))
    except:
        pass
X = np.vstack(all_patches)
pca = PCA(n_components=NODE_DIM, random_state=SEED)
pca.fit(X)
print("PCA fitted!")
builder = MRIGraphBuilder(pca=pca)

In [ ]:
# ---- Dataset class with Disk + RAM Caching ----
class AlzheimerGraphDataset(Dataset):
    def __init__(self, paths, labels, builder, split='train', cache_dir='/content/outputs/graph_cache'):
        self.paths = paths
        self.labels = labels
        self.builder = builder
        self.split = split
        self.cache_dir = cache_dir
        os.makedirs(cache_dir, exist_ok=True)
        self.mem_cache = [None] * len(paths)
        print(f"[{split}] {len(paths)} samples (caching to disk and RAM)")
        
    def __len__(self): 
        return len(self.paths)
        
    def __getitem__(self, idx):
        if self.mem_cache[idx] is not None:
            return self.mem_cache[idx]
            
        import hashlib
        path = self.paths[idx]
        label = self.labels[idx]
        hash_str = hashlib.md5(path.encode('utf-8')).hexdigest()
        cache_file = os.path.join(self.cache_dir, f"{self.split}_{hash_str}.pt")
        
        data = None
        if os.path.exists(cache_file):
            try:
                data = torch.load(cache_file)
            except:
                pass
        if data is None:
            data = self.builder.build(path, label)
            torch.save(data, cache_file)
            
        self.mem_cache[idx] = data
        return data

train_ds = AlzheimerGraphDataset(tr_p, tr_l, builder, 'train')
val_ds   = AlzheimerGraphDataset(va_p, va_l, builder, 'val')
test_ds  = AlzheimerGraphDataset(test_paths, test_labels, builder, 'test')

train_loader = GeoDataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = GeoDataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = GeoDataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

### 📊 Extract Features for traditional ML baselines

In [ ]:
def extract_features(loader, device):
    all_feats = []
    all_labels = []
    for batch in loader:
        batch = batch.to(device)
        # Global average pool over all 196 nodes per graph
        pooled = global_mean_pool(batch.x, batch.batch)
        all_feats.append(pooled.cpu().numpy())
        all_labels.append(batch.y.cpu().numpy())
    return np.vstack(all_feats), np.concatenate(all_labels)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Extracting features on: {device}")
X_train, y_train = extract_features(train_loader, device)
X_val, y_val = extract_features(val_loader, device)
X_test, y_test = extract_features(test_loader, device)

X_train_full = np.vstack([X_train, X_val])
y_train_full = np.concatenate([y_train, y_val])

print(f"X_train shape: {X_train_full.shape} | X_test shape: {X_test.shape}")

### 🤖 Train ML Classifiers

In [ ]:
classifiers = {
    "SVM": SVC(kernel='rbf', C=1.0, random_state=SEED),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=SEED),
    "XGBoost": XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=SEED, verbosity=0),
    "Bagging (DT)": BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=50, random_state=SEED),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, random_state=SEED),
    "LightGBM": LGBMClassifier(n_estimators=100, learning_rate=0.1, random_state=SEED, verbose=-1)
}

results = []

for name, clf in classifiers.items():
    print(f"Training {name}...")
    t0 = time.time()
    clf.fit(X_train_full, y_train_full)
    train_time = time.time() - t0
    
    y_pred = clf.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    
    print(f"  Accuracy: {acc:.4f} | F1: {f1:.4f}")
    results.append({
        "Model": name,
        "Test Accuracy": acc,
        "Macro Precision": prec,
        "Macro Recall": rec,
        "Macro F1-Score": f1
    })

### 📊 Compile Results table (Including GAT and CNN Baseline)

In [ ]:
# Set CNN metrics from the baseline outputs
results.append({
    "Model": "CNN Baseline",
    "Test Accuracy": 0.9373,
    "Macro Precision": 0.9385,
    "Macro Recall": 0.9373,
    "Macro F1-Score": 0.9379
})

# Set GAT metrics from Phase 2 outputs
results.append({
    "Model": "GAT (Ours)",
    "Test Accuracy": 0.6794,
    "Macro Precision": 0.6043,
    "Macro Recall": 0.8215,
    "Macro F1-Score": 0.6295
})

df_comparison = pd.DataFrame(results)

# Sorting to emphasize GAT and CNN
df_comparison['Sort_Key'] = df_comparison['Model'].apply(
    lambda x: 0 if 'GAT' in x else (1 if 'CNN' in x else 2)
)
df_comparison = df_comparison.sort_values(by=['Sort_Key', 'Test Accuracy'], ascending=[True, False]).drop(columns='Sort_Key')

# Save comparison sheets
df_comparison.to_csv('comparative_analysis.csv', index=False)
df_comparison.to_excel('comparative_analysis.xlsx', index=False)
print(df_comparison.to_string(index=False))

### 🖼️ Plot Comparative Metrics

In [ ]:
df_plot = df_comparison.melt(id_vars='Model', value_vars=['Test Accuracy', 'Macro F1-Score'],
                             var_name='Metric', value_name='Score')

plt.figure(figsize=(14, 7))
sns.set_theme(style="whitegrid")

custom_colors = {"Test Accuracy": "#2196F3", "Macro F1-Score": "#4CAF50"}

ax = sns.barplot(
    data=df_plot,
    x='Model',
    y='Score',
    hue='Metric',
    palette=custom_colors,
    edgecolor='black',
    linewidth=0.5
)

for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f"{height:.3f}",
                    (p.get_x() + p.get_width() / 2., height + 0.01),
                    ha='center', va='bottom', fontsize=8, color='#212121', fontweight='semibold')

plt.title("Comparative Performance Analysis: CNN vs. GAT vs. Machine Learning Baselines",
          fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Classifier Model", fontsize=11, labelpad=10)
plt.ylabel("Score", fontsize=11, labelpad=10)
plt.xticks(rotation=20, ha='right')
plt.ylim(0, 1.1)
plt.legend(loc='upper right', frameon=True, shadow=True)
plt.tight_layout()
plt.savefig('comparative_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

### 📥 Download Results

In [ ]:
files.download('comparative_analysis.xlsx')
files.download('comparative_metrics.png')